# Ming Image 0.1 Design on Kaggle (Wan2GP)

Single T4 setup. Heavy weights go in `/tmp`.

## 1. Install Wan2GP

In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
else:
    !git -C /kaggle/working/Wan2GP pull
%cd /kaggle/working/Wan2GP

## 2. Install requirements

In [ ]:
!pip install -r requirements.txt

## 3. Download heavy models to `/tmp`

Only the transformer and text encoder. VAE, vision tower and the small files are light; let Wan2GP download them.

### Transformer

In [ ]:
from huggingface_hub import hf_hub_download

base = hf_hub_download(
    repo_id='DeepBeepMeep/MingImage',
    filename='Ming-Image-0.1-Design_int8_convrot.safetensors',
    local_dir='/tmp/ming_image'
)
print('Transformer:', base)
!ls -lh /tmp/ming_image

### Text encoder

In [ ]:
encoder = hf_hub_download(
    repo_id='DeepBeepMeep/MingImage',
    filename='BailingMM2-Ming-Image/BailingMM2-Ming-Image-Core_int8_convrot.safetensors',
    local_dir='/tmp/ming_image'
)
print('Encoder:', encoder)
!du -sh /tmp/ming_image

## 4. Patch FlashAttention default

T4 can't run FlashAttention 2, but Ming's vision tower defaults to it. Re-run after every `git pull` in step 1.

In [ ]:
UP = '/kaggle/working/Wan2GP/models/ming_image/upstream'
!sed -i 's/_attn_implementation="flash_attention_2"/_attn_implementation="sdpa"/' {UP}/qwen2_5_vit.py
!sed -i 's/_attn_implementation="flash_attention_2"/_attn_implementation="eager"/' {UP}/configuration_bailing_moe_v2.py
!grep -n '_attn_implementation=' {UP}/qwen2_5_vit.py {UP}/configuration_bailing_moe_v2.py

## 5. Launch Wan2GP

Profile 5 is the low-VRAM profile. Only one T4 will be used.

In [ ]:
%cd /kaggle/working/Wan2GP

!python wgp.py --share

## 6. Select the model in Wan2GP

Pick **Ming Image 0.1 → Design** in the model dropdown, choose the **INT8** variants in the config, then click **Create Finetune**.

Files, use these:

### Main checkpoint
```
/tmp/ming_image/Ming-Image-0.1-Design_int8_convrot.safetensors
```

### Text encoder checkpoint
```
/tmp/ming_image/BailingMM2-Ming-Image/BailingMM2-Ming-Image-Core_int8_convrot.safetensors
```

## 7. Settings

| Setting | Design |
|---|---|
| Resolution | `1024x1024` |
| Steps | `12` |
| Guidance (CFG) | `1` |
| Number of images | `1` |
| Reference image | optional (one, for editing) |
| Acceleration LoRA | none |

## 8. Test prompt

**Prompt**
```
A clean mobile app UI screen for a coffee shop called "Bean There", header with the app name, three menu cards with prices, bottom navigation bar, warm brown and cream colors, flat modern design, crisp readable text.
```

## License

Ming-Image-0.1 is MIT licensed.